In [2]:
import shutil
from datasets import load_dataset

free_gb = shutil.disk_usage("/").free / 1e9
print(f"Free disk space: {free_gb:.1f} GB")
assert free_gb > 5, "Under 5GB free — clear space before continuing (see huggingface-cli delete-cache)"

hotpot = load_dataset(
    "hotpotqa/hotpot_qa",
    split="validation[:200]",
    revision="refs/convert/parquet",
)

nq = load_dataset(
    "google-research-datasets/nq_open",
    split="validation[:200]",
)

print("HotpotQA:", len(hotpot), "examples")
print("NQ:", len(nq), "examples")
print("\nHotpotQA sample:", hotpot[0])
print("\nNQ sample:", nq[0])

Free disk space: 8.1 GB
HotpotQA: 200 examples
NQ: 200 examples

HotpotQA sample: {'id': '5a8b57f25542995d1e6f1371', 'question': 'Were Scott Derrickson and Ed Wood of the same nationality?', 'answer': 'yes', 'type': 'comparison', 'level': 'hard', 'supporting_facts': {'title': ['Scott Derrickson', 'Ed Wood'], 'sent_id': [0, 0]}, 'context': {'title': ['Ed Wood (film)', 'Scott Derrickson', 'Woodson, Arkansas', 'Tyler Bates', 'Ed Wood', 'Deliver Us from Evil (2014 film)', 'Adam Collis', 'Sinister (film)', 'Conrad Brooks', 'Doctor Strange (2016 film)'], 'sentences': [['Ed Wood is a 1994 American biographical period comedy-drama film directed and produced by Tim Burton, and starring Johnny Depp as cult filmmaker Ed Wood.', " The film concerns the period in Wood's life when he made his best-known films as well as his relationship with actor Bela Lugosi, played by Martin Landau.", ' Sarah Jessica Parker, Patricia Arquette, Jeffrey Jones, Lisa Marie, and Bill Murray are among the supporting cas

In [3]:
import sys
sys.path.append("..")  # so we can import from the project root

import random
import json
from perturbations.generators import (
    inject_contradiction,
    inject_lexical_distractor,
    inject_stale_document,
    inject_multihop_gap,
    inject_topical_noise,
)

rng = random.Random(42)  # fixed seed for reproducibility
hotpot_list = list(hotpot)  # convert HF Dataset to plain list of dicts, easier to work with

PERTURBATIONS = {
    "contradiction": inject_contradiction,
    "lexical_distractor": inject_lexical_distractor,
    "stale_document": inject_stale_document,
    "multihop_gap": inject_multihop_gap,
    "topical_noise": inject_topical_noise,
}

stress_test_suite = []
for example in hotpot_list:
    for name, fn in PERTURBATIONS.items():
        perturbed = fn(example, corpus_pool=hotpot_list, rng=rng)
        stress_test_suite.append(perturbed)

print(f"Built {len(stress_test_suite)} stress-test examples "
      f"({len(hotpot_list)} originals x {len(PERTURBATIONS)} perturbation types)")

# Quick sanity check: print one example per perturbation type
seen_types = set()
for ex in stress_test_suite:
    if ex["perturbation_type"] not in seen_types:
        seen_types.add(ex["perturbation_type"])
        print(f"\n=== {ex['perturbation_type']} ===")
        print("Question:", ex["question"])
        print("Answer:", ex["answer"])
        print("Context titles:", ex["context"]["title"])

# Save to disk so later phases don't need to regenerate this
with open("../data/datasets/stress_test_suite.json", "w") as f:
    json.dump(stress_test_suite, f, indent=2)
print("\nSaved to data/datasets/stress_test_suite.json")

Built 1000 stress-test examples (200 originals x 5 perturbation types)

=== contradiction ===
Question: Were Scott Derrickson and Ed Wood of the same nationality?
Answer: yes
Context titles: ['Ed Wood (film)', 'Scott Derrickson', 'Woodson, Arkansas', 'Tyler Bates', 'Ed Wood', 'Deliver Us from Evil (2014 film)', 'Adam Collis', 'Sinister (film)', 'Conrad Brooks', 'Doctor Strange (2016 film)']

=== lexical_distractor ===
Question: Were Scott Derrickson and Ed Wood of the same nationality?
Answer: yes
Context titles: ['Ed Wood (film)', 'Scott Derrickson', 'Woodson, Arkansas', 'Tyler Bates', 'Ed Wood', 'Deliver Us from Evil (2014 film)', 'Adam Collis', 'Sinister (film)', 'Conrad Brooks', 'Doctor Strange (2016 film)', 'Gone in 60 Seconds (2000 film) (lexical distractor)']

=== stale_document ===
Question: Were Scott Derrickson and Ed Wood of the same nationality?
Answer: yes
Context titles: ['Ed Wood (film)', 'Scott Derrickson (2010 archive)', 'Woodson, Arkansas', 'Tyler Bates', 'Ed Wood', '